In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path
from collections import Counter
import hashlib

In [3]:
# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [4]:
dataset_path = r"D:\Project Jupyter\Dataset_murni"


In [5]:
# Inisialisasi list penampung
image_paths = []
labels = []
extensions = []
splits = []

# Daftar folder split
split_folders = ['train', 'val', 'test']

# Loop tiap split (train/val/test)
for split_name in split_folders:
    split_path = os.path.join(dataset_path, split_name)
    
    # Ambil daftar kelas dalam split (edible/poisonous)
    classes = sorted([d for d in os.listdir(split_path) if os.path.isdir(os.path.join(split_path, d))])
    
    for class_name in classes:
        class_path = os.path.join(split_path, class_name)
        
        # Ambil file gambar dengan ekstensi tertentu
        images = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        
        for img in images:
            image_paths.append(os.path.join(class_path, img))
            labels.append(class_name)
            splits.append(split_name)
            extensions.append(img.split('.')[-1].lower())


In [6]:
df = pd.DataFrame({
    'image_path': image_paths,
    'label': labels,
    'extension': extensions,
    'split': splits
})


In [11]:
# Hitung jenis file
ext_counts = df['extension'].value_counts()
print(f"Dataset berhasil dimuat!\n")
print(f" Total gambar: {len(df)}")
print(f" Jumlah kelas: {df['label'].nunique()}")
print(f" Nama kelas: {list(df['label'].unique())}")
print(f" Split : {list(df['split'].unique())}\n")

print(f" Jenis file gambar:")
for ext, count in ext_counts.items():
    percentage = (count / len(df)) * 100
    print(f"  - {ext}: {count} gambar ({percentage:.1f}%)")


Dataset berhasil dimuat!

 Total gambar: 2770
 Jumlah kelas: 2
 Nama kelas: ['edible', 'poisonous']
 Split : ['train', 'val', 'test']

 Jenis file gambar:
  - jpg: 2616 gambar (94.4%)
  - png: 154 gambar (5.6%)


In [13]:
all_files = []
for root, dirs, files in os.walk(dataset_path):
    for file in files:
        all_files.append(file)

# Hitung duplikat
duplicates = [item for item, count in Counter(all_files).items() if count > 1]

print("Jumlah file duplikat berdasarkan nama:", len(duplicates))
print("Daftar file duplikat:", duplicates)

Jumlah file duplikat berdasarkan nama: 0
Daftar file duplikat: []


In [15]:
import os
import hashlib

hashes = {}
duplicates = []

def hash_file(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

for root, dirs, files in os.walk(split_path):
    for file in files:
        file_path = os.path.join(root, file)
        file_hash = hash_file(file_path)

        if file_hash in hashes:
            duplicates.append(file_path)   # simpan yang mau dihapus
        else:
            hashes[file_hash] = file_path  # simpan yang pertama

print("Jumlah gambar duplikat:", len(duplicates))

# HAPUS FILE DUPLIKAT
for dup_path in duplicates:
    os.remove(dup_path)
    print("Dihapus:", dup_path)

print("Selesai hapus duplikat.")


Jumlah gambar duplikat: 0
Selesai hapus duplikat.


In [16]:
import os
import cv2
import pytesseract
from PIL import Image

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

text_images = []

for root, dirs, files in os.walk(split_path):
    for f in files:
        if f.lower().endswith((".jpg", ".png", ".jpeg")):
            path = os.path.join(root, f)

            img = cv2.imread(path)
            if img is None:
                continue

            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            text = pytesseract.image_to_string(gray)

            if len(text.strip()) > 10:   # threshold teks
                text_images.append(path)
                print("TEXT IMAGE:", path)

print("\nTOTAL GAMBAR MENGANDUNG TEKS:", len(text_images))



TOTAL GAMBAR MENGANDUNG TEKS: 0


In [17]:
import os
import cv2
import pytesseract

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

deleted = 0
skipped = 0

for root, dirs, files in os.walk(split_path):
    for f in files:
        if f.lower().endswith((".jpg", ".png", ".jpeg")):
            path = os.path.join(root, f)

            img = cv2.imread(path)
            if img is None:
                continue

            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            text = pytesseract.image_to_string(gray)

            if len(text.strip()) > 10:   # dianggap gambar teks
                try:
                    os.remove(path)
                    print("DELETED:", path)
                    deleted += 1
                except Exception as e:
                    print("FAILED:", path, "->", e)
                    skipped += 1

print("\nTOTAL DIHAPUS:", deleted)
print("TOTAL GAGAL:", skipped)



TOTAL DIHAPUS: 0
TOTAL GAGAL: 0
